# Group-split statistical experiment runner

This runner fits only `NBRFI` versus `None` with the corrected group split and metadata feature set `both`. `NoneWNBRFI` is loaded only after fitting for separate hard-negative evaluation; it never affects fitting, threshold selection, ordinary test metrics, or candidate choice.

Each candidate writes its own joblib bundle and independent manifest. This notebook records results without choosing a winner: the later comparison notebook reads manifests and makes the across-candidate comparison.

In [1]:
from __future__ import annotations

from copy import deepcopy
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from scipy.special import expit


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'rfimt').is_dir():
            return candidate
    raise RuntimeError('Could not locate the rfimt repository.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.experiments import load_experiment_spec, make_run_manifest, write_run_manifest
from rfimt.metrics import best_threshold_by_f1, eval_binary

CONFIG_PATH = REPO_ROOT / 'configs/experiments/b0531_group_split_statistical_both_v1.json'
spec = load_experiment_spec(CONFIG_PATH)
assert spec['model']['family'] == 'statistical'
assert spec['representation']['feature_set'] == 'both'
assert spec['labels']['hard_negative_role'] == 'evaluate_only'

DATASET, OUTPUT_DIR = spec['dataset'], Path(spec['outputs']['run_directory'])
FEATURES = spec['representation']['features']
POSITIVE, NEGATIVE, HARD_NEGATIVE = spec['labels']['positive'], spec['labels']['ordinary_negative'], spec['labels']['hard_negative']
GROUP_COLUMN, RANDOM_STATE = spec['split']['group_column'], spec['model']['random_state']


In [2]:
meta = pd.read_csv(DATASET['metadata_path'])
splits = np.load(DATASET['split_indices_path'])
split_indices = {name: splits[name] for name in ('train', 'val', 'test')}

missing = set(FEATURES).difference(meta.columns)
if missing or GROUP_COLUMN not in meta:
    raise ValueError(f'Metadata is missing columns: {sorted(missing | {GROUP_COLUMN})}')

# The corrected artifact is the sole fitting universe; fail rather than silently admitting hard negatives.
allowed_labels = {POSITIVE, NEGATIVE}
if not set(meta['label'].dropna().unique()).issubset(allowed_labels):
    raise ValueError('Fitting metadata contains labels outside NBRFI/None.')
for name, indices in split_indices.items():
    labels = set(meta.iloc[indices]['label'].dropna().unique())
    if not labels.issubset(allowed_labels):
        raise ValueError(f'{name} contains non-fitting labels: {labels - allowed_labels}')

# Group overlap would leak segment context into validation or test estimates.
groups = {name: set(meta.iloc[indices][GROUP_COLUMN]) for name, indices in split_indices.items()}
if groups['train'] & groups['val'] or groups['train'] & groups['test'] or groups['val'] & groups['test']:
    raise ValueError('Corrected split has overlapping segment groups.')

X = meta[FEATURES].apply(pd.to_numeric, errors='coerce')
y = meta['label'].eq(POSITIVE).astype(int).to_numpy()
X_train, X_val, X_test = (X.iloc[split_indices[name]] for name in ('train', 'val', 'test'))
y_train, y_val, y_test = (y[split_indices[name]] for name in ('train', 'val', 'test'))

hard_meta = pd.read_csv(DATASET['hard_negative_metadata_path'])
hard_source_indices = np.load(DATASET['hard_negative_source_indices_path'])
if len(hard_meta) != len(hard_source_indices) or not hard_meta['label'].eq(HARD_NEGATIVE).all():
    raise ValueError('Hard-negative artifacts are inconsistent.')
missing_hard = set(FEATURES).difference(hard_meta.columns)
if missing_hard:
    raise ValueError(f'Hard-negative metadata is missing features: {sorted(missing_hard)}')
X_hard = hard_meta[FEATURES].apply(pd.to_numeric, errors='coerce')
print({name: len(indices) for name, indices in split_indices.items()}, 'hard_negative=', len(X_hard))


{'train': 16000, 'val': 2000, 'test': 2000} hard_negative= 1312753


In [3]:
def positive_scores(model, features):
    classes = np.asarray(model.classes_)
    if not np.array_equal(classes, np.array([0, 1])):
        raise ValueError(f'Expected binary classes [0, 1], got {classes.tolist()}.')
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1], 'predict_proba'
    if hasattr(model, 'decision_function'):
        # Monotonic score mapping supports threshold/ranking metrics, not probability calibration.
        return expit(model.decision_function(features)), 'decision_function_sigmoid'
    raise TypeError('Statistical candidate exposes neither predict_proba nor decision_function.')


def segment_rows(candidate, test_meta, probability, threshold):
    rows = []
    for segment, positions in test_meta.groupby(GROUP_COLUMN, sort=True).groups.items():
        positions = np.asarray(list(positions))
        local_y, local_p = y_test[positions], probability[positions]
        tn, fp, fn, tp = confusion_matrix(local_y, local_p >= threshold, labels=[0, 1]).ravel()
        predicted_positive_fraction = float((local_p >= threshold).mean())
        rows.append({'candidate': candidate, 'segment_index': segment, 'n_rows': len(local_y),
                     'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                     'mean_positive_score': float(local_p.mean()),
                     'predicted_positive_fraction': predicted_positive_fraction})
    return rows


def build_models():
    # Preserve the legacy model contract while fitting transforms on train rows only.
    imputer = [('imputer', SimpleImputer(strategy='constant', fill_value=0.0))]
    return {
        'SGD_LogReg': Pipeline(imputer + [('classifier', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2000, tol=1e-3, random_state=RANDOM_STATE))]),
        'SGD_LinearSVM': Pipeline(imputer + [('classifier', SGDClassifier(loss='hinge', alpha=1e-4, max_iter=2000, tol=1e-3, random_state=RANDOM_STATE))]),
        'Poly2_LogReg': Pipeline(imputer + [('poly2', PolynomialFeatures(degree=2, include_bias=False)), ('classifier', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2000, tol=1e-3, random_state=RANDOM_STATE))]),
        'RBFapprox_LogReg': Pipeline(imputer + [('rbf', RBFSampler(gamma=1.0, n_components=800, random_state=RANDOM_STATE)), ('classifier', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2000, tol=1e-3, random_state=RANDOM_STATE))]),
        'HistGB': Pipeline(imputer + [('classifier', HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=300, random_state=RANDOM_STATE))]),
        'MLP': Pipeline(imputer + [('classifier', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', alpha=1e-4, learning_rate_init=1e-3, max_iter=200, random_state=RANDOM_STATE))]),
    }


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
test_meta = meta.iloc[split_indices['test']].reset_index(drop=True)
code_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
results, per_segment, hard_negative_results = [], [], []

models = build_models()
declared_candidates = [item['name'] for item in spec['model']['candidates']]
if list(models) != declared_candidates:
    raise ValueError('Model implementation and declared candidate order disagree.')

# tqdm exposes progress while candidates remain independent rather than forming a test-driven model search.
for candidate, model in tqdm(models.items(), desc='Fitting candidates', mininterval=5.0):
    model.fit(X_train, y_train)
    validation_scores, score_source = positive_scores(model, X_val)
    threshold, validation_f1 = best_threshold_by_f1(y_val, validation_scores)

    # Test probabilities are generated once after this candidate's validation threshold is frozen.
    test_probability, test_score_source = positive_scores(model, X_test)
    if test_score_source != score_source:
        raise RuntimeError('The candidate changed score semantics after fitting.')
    test_metrics = eval_binary(y_test, test_probability, threshold)
    if score_source == 'decision_function_sigmoid':
        test_metrics['logloss'] = None
    hard_probability, hard_score_source = positive_scores(model, X_hard)
    if hard_score_source != score_source:
        raise RuntimeError('The candidate changed score semantics for hard negatives.')
    hard_predicted = hard_probability >= threshold

    candidate_dir = OUTPUT_DIR / candidate
    candidate_dir.mkdir()
    bundle_path = candidate_dir / 'model.joblib'
    joblib.dump({'model': model, 'threshold': threshold, 'features': FEATURES, 'spec_path': str(CONFIG_PATH)}, bundle_path)

    row = {'candidate': candidate, 'score_source': score_source, 'validation_f1': validation_f1, 'threshold': threshold, **test_metrics}
    results.append(row)
    per_segment.extend(segment_rows(candidate, test_meta, test_probability, threshold))
    hard_metrics = {'candidate': candidate, 'n_rows': int(len(hard_probability)),
                    'false_positive_count': int(hard_predicted.sum()), 'false_positive_rate': float(hard_predicted.mean()),
                    'mean_positive_score': float(hard_probability.mean()),
                    'median_positive_score': float(np.median(hard_probability)), 'threshold': threshold}
    hard_negative_results.append(hard_metrics)

    candidate_spec = deepcopy(spec)
    candidate_spec['experiment_id'] = f"{spec['experiment_id']}__{candidate.lower()}"
    candidate_spec['model'] = {'family': 'statistical', 'name': candidate, 'random_state': RANDOM_STATE, 'score_source': score_source}
    candidate_spec['outputs']['run_directory'] = str(candidate_dir)
    manifest = make_run_manifest(
        candidate_spec, metrics={'validation': {'f1': validation_f1, 'threshold': threshold}, 'row_test': test_metrics, 'hard_negative': hard_metrics},
        code_revision=code_revision,
        artifacts={'model_bundle': str(bundle_path), 'results_csv': str(OUTPUT_DIR / 'results.csv'),
                   'segment_metrics_csv': str(OUTPUT_DIR / 'segment_metrics.csv'),
                   'hard_negative_metrics_csv': str(OUTPUT_DIR / 'hard_negative_metrics.csv')},
        notes=['Candidate is independently recorded; comparison notebook decides later.',
               'NoneWNBRFI is evaluate-only and excluded from fit and threshold selection.'] + (['SGD_LinearSVM uses sigmoid-mapped decision scores for ranking and threshold selection; logloss is intentionally not reported because these are not calibrated probabilities.'] if score_source == 'decision_function_sigmoid' else []))
    manifest['candidate'] = candidate
    write_run_manifest(candidate_dir / 'manifest.json', manifest)

pd.DataFrame(results).to_csv(OUTPUT_DIR / 'results.csv', index=False)
pd.DataFrame(per_segment).to_csv(OUTPUT_DIR / 'segment_metrics.csv', index=False)
pd.DataFrame(hard_negative_results).to_csv(OUTPUT_DIR / 'hard_negative_metrics.csv', index=False)
pd.DataFrame(results)


Fitting candidates: 100%|██████████| 6/6 [00:31<00:00,  5.27s/it]


,candidate,score_source,validation_f1,threshold,TN,FP,FN,TP,accuracy,precision,recall,f1,roc_auc,pr_auc,logloss
0,SGD_LogReg,predict_proba,0.903614,0.02,917,83,104,896,0.9065,0.915220,0.896,0.905508,0.918199,0.887092,3.310823
1,SGD_LinearSVM,decision_function_sigmoid,0.863158,0.84,879,121,115,885,0.8820,0.879722,0.885,0.882353,0.889165,0.846250,NaN
2,Poly2_LogReg,predict_proba,0.895493,0.01,885,115,126,874,0.8795,0.883721,0.874,0.878834,0.879500,0.835372,4.343260
3,RBFapprox_LogReg,predict_proba,0.666889,0.23,0,1000,1,999,0.4995,0.499750,0.999,0.666222,0.492897,0.493951,0.711900
4,HistGB,predict_proba,0.993988,0.18,989,11,8,992,0.9905,0.989033,0.992,0.990514,0.999466,0.999501,0.026022
5,MLP,predict_proba,0.969419,0.52,986,14,57,943,0.9645,0.985371,0.943,0.963720,0.988712,0.991796,0.105808
